# Install and import necessary libraries

In [ ]:
%%capture
!pip install pyvi transformers transformers[torch] evaluate
!git clone --single-branch --branch fast_tokenizers_BARTpho_PhoBERT_BERTweet https://github.com/datquocnguyen/transformers.git
cd transformers
!pip3 install -e .

In [ ]:
from huggingface_hub import login
login('your_huggingface_auth_token_here')

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, pipeline

base_checkpoint = "vinai/phobert-base-v2"
custom_checkpoint = "./models/ViLegalBERT"

tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(custom_checkpoint)

print("Tokenizer và mô hình đã được tải thành công!")

In [ ]:
tokenizer.is_fast

In [ ]:
import evaluate
import torch
import collections
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from pyvi import ViTokenizer
from datasets import Dataset
import wandb

In [ ]:
wandb.login(key='your_wandb_auth_token_here')

In [ ]:
metric = evaluate.load("squad")

# Read the dataset

In [ ]:
splits = {'train': 'ViBiDLQA_train.csv', 'test': 'ViBiDLQA_test.csv'}
df_train = pd.read_csv("hf://datasets/ntphuc149/ViBidLQA/" + splits["train"])
df_test = pd.read_csv("hf://datasets/ntphuc149/ViBidLQA/" + splits["test"])

In [ ]:
df_train.columns, df_test.columns

In [ ]:
len(df_train), len(df_test)

# Data Preprocessing

In [ ]:
df_train['context'] = df_train['context'].astype(str)
df_train['question'] = df_train['question'].astype(str)
df_train['extractive_answer'] = df_train['extractive_answer'].astype(str)

In [ ]:
df_test['context'] = df_test['context'].astype(str)
df_test['question'] = df_test['question'].astype(str)
df_test['extractive_answer'] = df_test['extractive_answer'].astype(str)

In [ ]:
df_train['context'] = df_train['context'].apply(ViTokenizer.tokenize)
df_train['question'] = df_train['question'].apply(ViTokenizer.tokenize)
df_train['extractive_answer'] = df_train['extractive_answer'].apply(ViTokenizer.tokenize)

df_test['context'] = df_test['context'].apply(ViTokenizer.tokenize)
df_test['question'] = df_test['question'].apply(ViTokenizer.tokenize)
df_test['extractive_answer'] = df_test['extractive_answer'].apply(ViTokenizer.tokenize)

df_train = df_train[["context", "question", "extractive_answer"]]
df_train.columns = ["context", "question", "answer"]

df_test = df_test[["context", "question", "extractive_answer"]]
df_test.columns = ["context", "question", "answer"]

In [ ]:
def find_start_index(context, answer):
    return str(context).find(str(answer))

In [ ]:
df_train['start_index'] = df_train.apply(lambda row: find_start_index(context=row['context'], answer=row['answer']), axis=1)
df_test['start_index'] = df_test.apply(lambda row: find_start_index(context=row['context'], answer=row['answer']), axis=1)

In [ ]:
df_train[df_train['start_index'] == -1]

In [ ]:
df_test[df_test['start_index'] == -1]

In [ ]:
dataset_train = Dataset.from_pandas(df_train, preserve_index=False)

In [ ]:
dataset_train.column_names

In [ ]:
dataset_temp_train = []
for i in dataset_train:
    sample = {}
    sample['context'] = i['context']
    sample['question'] = i['question']
    sample['answer'] = {'text': [i['answer']], 'answer_start': [i['start_index']]}
    dataset_temp_train.append(sample)
    
df_train = pd.DataFrame(dataset_temp_train)
df_train

In [ ]:
dataset_temp_test = []
for i in dataset_test:
    sample = {}
    sample['context'] = i['context']
    sample['question'] = i['question']
    sample['answer'] = {'text': [i['answer']], 'answer_start': [i['start_index']]}
    dataset_temp_test.append(sample)
    
df_test = pd.DataFrame(dataset_temp_test)
df_test

In [ ]:
num_of_val_sample = 500

df_val = df_train.sample(n=num_of_val_sample, random_state=42)
df_train = df_train.drop(index=df_val.index)

print(f'Total samples in training set: {len(df_train)}')
print(f'Total samples in validation set: {len(df_val)}')
print(f'Total samples in test set: {len(df_test)}')

In [ ]:
train_set = Dataset.from_pandas(df_train, preserve_index=False)
val_set = Dataset.from_pandas(df_val, preserve_index=False)
test_set = Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
max_length = 256
stride = 8

def preprocess_training_examples(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        max_length= max_length,
        truncation="only_second",
        stride= stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answer"]
    start_positions = []
    end_positions = []
    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
train_dataset = train_set.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=train_set.column_names,
)
len(train_set), len(train_dataset)

In [ ]:
def preprocess_validation_test_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["question"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [ ]:
validation_dataset = val_set.map(
    preprocess_validation_test_examples,
    batched=True,
    remove_columns=val_set.column_names,
)
len(val_set), len(validation_dataset)

In [ ]:
test_dataset = test_set.map(
    preprocess_validation_test_examples,
    batched=True,
    remove_columns=test_set.column_names,
)
len(test_set), len(test_dataset)

In [ ]:
n_best = 50
max_answer_length = 512

def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["question"]
        context = example["context"]
        answers = []

        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["question"], "answers": ex["answer"]} for ex in examples]
    
    for i, ii in zip(predicted_answers, theoretical_answers):
        print("-"*99)
        print(f"Pred: {i['prediction_text']}")
#         print("*"*20)
        print(f"Goal: {ii['answers']['text'][0]}")
    
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
num_batch = 2
learning_rate = 2e-5
num_epochs = 5

In [ ]:
args = TrainingArguments(
    "bert-finetuned-squad",
    evaluation_strategy="no",
    save_strategy = "no",
    eval_steps=500,
    save_strategy="epoch",
    save_total_limit = 1,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    per_device_train_batch_size=num_batch,
    per_device_eval_batch_size=num_batch,
    learning_rate= learning_rate,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    fp16=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
predictions, _, _ = trainer.predict(test_dataset)
start_logits, end_logits = predictions

compute_metrics(start_logits, end_logits, test_dataset, test_set)